In [9]:
import os
import json
import pymysql
from dotenv import load_dotenv, find_dotenv
from kafka import KafkaConsumer

# Automatically find and load the .env file from the parent/root directory
load_dotenv(find_dotenv())

# Get Kafka connection settings
bootstrap_servers = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")
topic_name = "tech-challenge.events"

# Get database connection settings
db_host = os.getenv("DB_HOST", "localhost")
db_port = int(os.getenv("DB_PORT", 3306))
db_user = os.getenv("DB_USER", "root")
db_password = os.getenv("DB_PASSWORD", "password")
db_database = os.getenv("DB_DATABASE", "tech")

# Custom deserializer to parse JSON values, returning None if parsing fails
def deserialize_value(value):
    if value is None:
        return None
    try:
        return json.loads(value.decode("utf-8"))
    except Exception:
        return None

# Initialize Kafka consumer
print(f"Connecting to Kafka at {bootstrap_servers}...")
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=bootstrap_servers.split(","),
    auto_offset_reset="earliest",  # Start reading from the beginning
    enable_auto_commit=True,
    value_deserializer=deserialize_value
)

# Connect to the database
print(f"Connecting to database '{db_database}' on '{db_host}:{db_port}' as '{db_user}'...")
db_connection = pymysql.connect(
    host=db_host,
    port=db_port,
    user=db_user,
    password=db_password,
    database=db_database
)

print(f"Listening for events on topic '{topic_name}' (Press Stop/Interrupt to exit)...")

insert_query = """
INSERT INTO br_inep_avaliacao_alfabetizacao_uf 
(ano, sigla_uf, serie, rede, taxa_alfabetizacao, media_portugues, 
 proporcao_aluno_nivel_0, proporcao_aluno_nivel_1, proporcao_aluno_nivel_2, 
 proporcao_aluno_nivel_3, proporcao_aluno_nivel_4, proporcao_aluno_nivel_5, 
 proporcao_aluno_nivel_6, proporcao_aluno_nivel_7, proporcao_aluno_nivel_8) 
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

try:
    for message in consumer:
        event_data = message.value
        
        # Ensure we got a valid JSON dictionary
        if not isinstance(event_data, dict):
            print(f"Skipping invalid/non-JSON message format at offset {message.offset}")
            continue
            
        # Validate that the minimum required data is present and not None
        required_fields = ["ano", "sigla_uf", "serie", "rede", "taxa_alfabetizacao", "media_portugues"]
        if not all(event_data.get(field) is not None for field in required_fields):
            print(f"Skipping event at offset {message.offset} due to missing required fields: {event_data}")
            continue
            
        try:
            # Extract and parse values according to DB column types
            ano = int(event_data.get("ano")) if event_data.get("ano") is not None else None
            sigla_uf = str(event_data.get("sigla_uf")) if event_data.get("sigla_uf") is not None else None
            serie = int(event_data.get("serie")) if event_data.get("serie") is not None else None
            rede = int(event_data.get("rede")) if event_data.get("rede") is not None else None
            taxa_alfabetizacao = float(event_data.get("taxa_alfabetizacao")) if event_data.get("taxa_alfabetizacao") is not None else None
            media_portugues = float(event_data.get("media_portugues")) if event_data.get("media_portugues") is not None else None
            
            # Additional nullable columns
            p0 = int(event_data.get("proporcao_aluno_nivel_0")) if event_data.get("proporcao_aluno_nivel_0") is not None else None
            p1 = float(event_data.get("proporcao_aluno_nivel_1")) if event_data.get("proporcao_aluno_nivel_1") is not None else None
            p2 = float(event_data.get("proporcao_aluno_nivel_2")) if event_data.get("proporcao_aluno_nivel_2") is not None else None
            p3 = float(event_data.get("proporcao_aluno_nivel_3")) if event_data.get("proporcao_aluno_nivel_3") is not None else None
            p4 = float(event_data.get("proporcao_aluno_nivel_4")) if event_data.get("proporcao_aluno_nivel_4") is not None else None
            p5 = float(event_data.get("proporcao_aluno_nivel_5")) if event_data.get("proporcao_aluno_nivel_5") is not None else None
            p6 = float(event_data.get("proporcao_aluno_nivel_6")) if event_data.get("proporcao_aluno_nivel_6") is not None else None
            p7 = float(event_data.get("proporcao_aluno_nivel_7")) if event_data.get("proporcao_aluno_nivel_7") is not None else None
            p8 = float(event_data.get("proporcao_aluno_nivel_8")) if event_data.get("proporcao_aluno_nivel_8") is not None else None
            
            # Execute database insertion
            with db_connection.cursor() as cursor:
                cursor.execute(insert_query, (ano, sigla_uf, serie, rede, taxa_alfabetizacao, media_portugues, p0, p1, p2, p3, p4, p5, p6, p7, p8))
            db_connection.commit()
            
            print(f"Successfully inserted event (Offset: {message.offset}) -> Year: {ano}, UF: {sigla_uf}, Rate: {taxa_alfabetizacao}")
            
        except Exception as insert_err:
            print(f"Failed to process message at offset {message.offset}: {insert_err}")
            db_connection.rollback()
            
except KeyboardInterrupt:
    print("\nStopping consumer stream...")
finally:
    consumer.close()
    db_connection.close()
    print("Connections closed. Consumer stopped successfully.")

Connecting to Kafka at localhost:9092...
Connecting to database 'tech' on 'localhost:3306' as 'root'...
Listening for events on topic 'tech-challenge.events' (Press Stop/Interrupt to exit)...


C:\Users\thiag\AppData\Local\Temp\ipykernel_27124\2389699488.py:32: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Successfully inserted event (Offset: 0) -> Year: 2027, UF: SP, Rate: 85.5
Successfully inserted event (Offset: 1) -> Year: 2028, UF: SP, Rate: 85.5
Successfully inserted event (Offset: 2) -> Year: 2029, UF: RJ, Rate: 85.5

Stopping consumer stream...
Connections closed. Consumer stopped successfully.
